# Task 17 Source/Value Column Profile

목적: 유전체/분자병리 정보를 찾기 전에, source/value 후보 column의 값 형태를 먼저 확인합니다.

확인 순서:

1. 후보 column 전체 profile
2. column별 형태 판정: local code / numeric string / short categorical / long text
3. long text 또는 marker가 직접 들어갈 가능성이 있는 column에만 regex 적용
4. code형 column은 concept_id 또는 source code mapping 확인

대상 column:

- `measurement.measurement_source_value`
- `measurement.value_source_value`
- `procedure_occurrence.procedure_source_value`
- `observation.observation_source_value`
- `observation.value_source_value`

In [ ]:
import os
import getpass
import pandas as pd
import psycopg

DB_HOST = os.environ.get("SNUH_CDM_HOST", "pg-2vge6u.vpc-cdb-kr.gov-ntruss.com")
DB_PORT = int(os.environ.get("SNUH_CDM_PORT", "5432"))
DB_NAME = os.environ.get("SNUH_CDM_DATABASE", "cdm")
DB_USER = os.environ.get("SNUH_CDM_USER", "jaegyun_jung")
DB_SCHEMA = os.environ.get("SNUH_CDM_SCHEMA", "cdm2024_official")
DB_SSLMODE = os.environ.get("SNUH_CDM_SSLMODE", "disable")
DB_END_DATE = "2025-02-05"

# 1 means 0.1% because the sampling denominator below is 1000.
SOURCE_SAMPLE_PER_MILLE = 1
SOURCE_SAMPLE_SEED = 20260617

# This is a profile notebook. Use a longer timeout than the earlier 3-5 min checks.
STATEMENT_TIMEOUT = "30min"

password = os.environ.get("SNUH_CDM_PASSWORD")
if not password:
    password = getpass.getpass("SNUH CDM password: ")

conn = psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=password,
    sslmode=DB_SSLMODE,
    application_name="task17_source_value_column_profile",
    options=f"-c statement_timeout={STATEMENT_TIMEOUT}",
)
conn.autocommit = True

def run(sql_text, params=()):
    with conn.cursor() as cur:
        cur.execute(sql_text, params)

def q(sql_text, params=()):
    with conn.cursor() as cur:
        cur.execute(sql_text, params)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

pd.set_option("display.max_colwidth", 240)
pd.set_option("display.max_rows", 100)

## 1. Create Patient Sample

0.1% 환자 표본을 만듭니다. 전체 count가 아니라 column 형태 확인용입니다.

In [ ]:
run("DROP TABLE IF EXISTS tmp_task17_profile_person")
run(f"""
CREATE TEMP TABLE tmp_task17_profile_person
ON COMMIT PRESERVE ROWS AS
SELECT person_id
FROM {DB_SCHEMA}.person
WHERE mod(
    hashtextextended(person_id::text, %s) & 9223372036854775807,
    1000
) < %s
""", (SOURCE_SAMPLE_SEED, SOURCE_SAMPLE_PER_MILLE))
run("CREATE INDEX ON tmp_task17_profile_person(person_id)")
run("ANALYZE tmp_task17_profile_person")

display(q("SELECT COUNT(*)::bigint AS sampled_patients FROM tmp_task17_profile_person"))

In [ ]:
COLUMN_SPECS = [
    {
        "label": "measurement.measurement_source_value",
        "table": "measurement",
        "date_col": "measurement_date",
        "value_col": "measurement_source_value",
        "id_cols": ["measurement_concept_id", "measurement_source_concept_id", "value_as_concept_id"],
    },
    {
        "label": "measurement.value_source_value",
        "table": "measurement",
        "date_col": "measurement_date",
        "value_col": "value_source_value",
        "id_cols": ["measurement_concept_id", "measurement_source_concept_id", "value_as_concept_id"],
    },
    {
        "label": "procedure_occurrence.procedure_source_value",
        "table": "procedure_occurrence",
        "date_col": "procedure_date",
        "value_col": "procedure_source_value",
        "id_cols": ["procedure_concept_id", "procedure_source_concept_id"],
    },
    {
        "label": "observation.observation_source_value",
        "table": "observation",
        "date_col": "observation_date",
        "value_col": "observation_source_value",
        "id_cols": ["observation_concept_id", "observation_source_concept_id", "value_as_concept_id"],
    },
    {
        "label": "observation.value_source_value",
        "table": "observation",
        "date_col": "observation_date",
        "value_col": "value_source_value",
        "id_cols": ["observation_concept_id", "observation_source_concept_id", "value_as_concept_id"],
    },
]

def column_exists(table, col):
    frame = q(
        """
        SELECT 1
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s AND column_name = %s
        LIMIT 1
        """,
        (DB_SCHEMA, table, col),
    )
    return len(frame) > 0

available_specs = []
for spec in COLUMN_SPECS:
    if column_exists(spec["table"], spec["value_col"]):
        available_specs.append(spec)
    else:
        print("missing", spec["label"])

available_specs

## 2. First Non-Null Examples

집계하지 않고 각 column의 non-null 예시만 봅니다. 여기서 code형인지 numeric string인지 long text인지 먼저 판단합니다.

In [ ]:
def first_examples(spec, limit=30):
    table = spec["table"]
    date_col = spec["date_col"]
    value_col = spec["value_col"]
    id_cols = [c for c in spec["id_cols"] if column_exists(table, c)]
    selected_ids = ",\n      ".join(id_cols) if id_cols else "NULL::text AS no_id_columns"
    return q(f"""
    SELECT
      {value_col}::text AS value,
      length({value_col}::text) AS value_len,
      {selected_ids},
      {date_col}::text AS event_date
    FROM {DB_SCHEMA}.{table} t
    JOIN tmp_task17_profile_person p USING (person_id)
    WHERE {date_col} BETWEEN DATE '1900-01-01' AND %s::date
      AND {value_col} IS NOT NULL
      AND NULLIF({value_col}::text, '') IS NOT NULL
    LIMIT %s
    """, (DB_END_DATE, limit))

example_frames = {}
for spec in available_specs:
    print("\n===", spec["label"], "===")
    df = first_examples(spec, limit=30)
    example_frames[spec["label"]] = df
    display(df)

## 3. Lightweight Shape Profile

`COUNT(DISTINCT)`는 일부 column에서 무거울 수 있으므로 쓰지 않습니다. non-null 수, 길이, 숫자처럼 보이는 값 비율만 봅니다.

In [ ]:
def lightweight_profile(spec):
    table = spec["table"]
    date_col = spec["date_col"]
    value_col = spec["value_col"]
    return q(f"""
    SELECT
      %s::text AS source_column,
      COUNT(*)::bigint AS sampled_rows,
      COUNT(*) FILTER (
        WHERE {value_col} IS NOT NULL
          AND NULLIF({value_col}::text, '') IS NOT NULL
      )::bigint AS nonnull_rows,
      AVG(length({value_col}::text)) FILTER (
        WHERE {value_col} IS NOT NULL
          AND NULLIF({value_col}::text, '') IS NOT NULL
      ) AS avg_len,
      MAX(length({value_col}::text)) FILTER (
        WHERE {value_col} IS NOT NULL
          AND NULLIF({value_col}::text, '') IS NOT NULL
      ) AS max_len,
      COUNT(*) FILTER (
        WHERE {value_col}::text ~ '^[+-]?[0-9]+(\\.[0-9]+)?$'
      )::bigint AS numeric_like_rows,
      COUNT(*) FILTER (
        WHERE length({value_col}::text) >= 50
      )::bigint AS long_text_like_rows
    FROM {DB_SCHEMA}.{table} t
    JOIN tmp_task17_profile_person p USING (person_id)
    WHERE {date_col} BETWEEN DATE '1900-01-01' AND %s::date
    """, (spec["label"], DB_END_DATE))

profiles = []
for spec in available_specs:
    print("profiling", spec["label"])
    profiles.append(lightweight_profile(spec))

profile_df = pd.concat(profiles, ignore_index=True)
profile_df["numeric_like_rate"] = profile_df["numeric_like_rows"] / profile_df["nonnull_rows"].replace({0: pd.NA})
profile_df["long_text_like_rate"] = profile_df["long_text_like_rows"] / profile_df["nonnull_rows"].replace({0: pd.NA})
display(profile_df)

## 4. Manual Shape Classification

위 예시와 profile을 보고 아래 셀의 값을 수동으로 채웁니다.

권장 label:

- `local_code`
- `numeric_string`
- `short_categorical`
- `long_text`
- `empty_or_unknown`

`long_text` 또는 marker가 직접 들어갈 가능성이 있는 column에만 다음 regex cell을 실행합니다.

In [ ]:
shape_decision = pd.DataFrame([
    {"source_column": "measurement.measurement_source_value", "shape": "", "next_method": ""},
    {"source_column": "measurement.value_source_value", "shape": "", "next_method": ""},
    {"source_column": "procedure_occurrence.procedure_source_value", "shape": "", "next_method": ""},
    {"source_column": "observation.observation_source_value", "shape": "", "next_method": ""},
    {"source_column": "observation.value_source_value", "shape": "", "next_method": ""},
])
display(shape_decision)

## 5. Optional Regex Check Only For Text-Like Columns

아래 `TEXT_LIKE_COLUMNS`에 직접 marker가 들어갈 가능성이 있는 column만 넣고 실행합니다. code형/numeric형 column은 넣지 않습니다.

In [ ]:
TEXT_LIKE_COLUMNS = [
    # Example: "observation.value_source_value"
]

MARKER_REGEX = r"\\m(EGFR|KRAS|NRAS|BRAF|ALK|ROS1|BRCA1|BRCA2|ERBB2|HER2|MSI|MMR|MLH1|MSH2|MSH6|PMS2|NTRK|RET|MET|PIK3CA|PD-L1|PDL1|NGS|mutation|fusion|rearrangement|amplification)\\M"

def marker_examples(spec, limit=50):
    table = spec["table"]
    date_col = spec["date_col"]
    value_col = spec["value_col"]
    id_cols = [c for c in spec["id_cols"] if column_exists(table, c)]
    selected_ids = ",\n      ".join(id_cols) if id_cols else "NULL::text AS no_id_columns"
    return q(f"""
    SELECT
      %s::text AS source_column,
      {value_col}::text AS value,
      length({value_col}::text) AS value_len,
      {selected_ids},
      {date_col}::text AS event_date
    FROM {DB_SCHEMA}.{table} t
    JOIN tmp_task17_profile_person p USING (person_id)
    WHERE {date_col} BETWEEN DATE '1900-01-01' AND %s::date
      AND {value_col} IS NOT NULL
      AND NULLIF({value_col}::text, '') IS NOT NULL
      AND {value_col}::text ~* %s
    LIMIT %s
    """, (spec["label"], DB_END_DATE, MARKER_REGEX, limit))

for label in TEXT_LIKE_COLUMNS:
    spec = next(s for s in available_specs if s["label"] == label)
    print("\n=== marker examples:", label, "===")
    display(marker_examples(spec, limit=50))